В этом ноутбуке мы будем работать над моделью предсказания движеняи цены акций по новостям, чтобы выявить важность признаков, а также актуальность бизнес-задачи.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("finhub+theguardian_raw_7.5k.csv", index_col=0)
df = df.dropna(subset=["fields.trailText"])
df.shape

В нашем датасете есть новости для которых есть краткая сводка, а есть новости для которых есть и краткая сводка и полный текст статьи. Для удобства будет обозначать их как short и full (или же s и f)

In [ ]:
df["fields.trailText"]

In [ ]:
df["fields.bodyText"]

Ячейками ниже преобразуем наши текста в эмбеддинги. Код ниже был запущен в ноутубке на kaggle, поскольку на CPU данная операция работает очень медленно. Код был перенесен сюда и закомментирован, чтобы он не выполнялся при нажатии на "Run All".

In [ ]:
# from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("intfloat/multilingual-e5-small")

# texts = ["passage: " + x for x in df["fields.bodyText"]]

# embeddings_full = model.encode(
#     texts,
#     normalize_embeddings=True,
#     show_progress_bar=True
# )

In [ ]:
# model = SentenceTransformer("intfloat/multilingual-e5-small")

# texts = ["passage: " + x for x in df["fields.trailText"]]

# embeddings_short = model.encode(
#     texts,
#     normalize_embeddings=True,
#     show_progress_bar=True
# )


Сохраняем наши эмбеддинги в файлы

In [ ]:
# np.save("embeddings_f.npy", embeddings_full)

In [ ]:
# np.save("embeddings_s.npy", embeddings_short)

Сразу же подгрузим и преобразуем в DataFrame

In [ ]:
np_embeddings_full = np.load('embeddings_f.npy')
np_embeddings_short = np.load('embeddings_s.npy')

In [ ]:
embeddings_full = pd.DataFrame(np_embeddings_full)
embeddings_short = pd.DataFrame(np_embeddings_short)

In [ ]:
embeddings_full.shape

In [ ]:

embeddings_short.shape

Соединим эмбеддинги с исходным датасетом, из которого оставим только то что пригодится для нашей модели - дата публикации, тикеры которые были указаны экспертами в статье, компании для которых эта новость была найдена, и источник новости.

In [ ]:
embeddings_full.columns = [f"emb_f_{i}" for i in range(embeddings_full.shape[1])]
embeddings_short.columns = [f"emb_s_{i}" for i in range(embeddings_short.shape[1])]

In [ ]:
df = df[["webPublicationDate", "stocks", "company", "source"]]

In [ ]:
data = pd.concat(
    [
        df.reset_index(drop=True),
        embeddings_full.reset_index(drop=True),
        embeddings_short.reset_index(drop=True)
    ],
    axis=1
)

В столбце company для статей из FinHub есть пропуски, зато в тикерах пропусков нет. Восстановим компании по тикерам:

In [ ]:
mapping = {
    "AAPL": "Apple",
    "TSLA": "Tesla",
    "CAT": "Caterpillar",
    "JPM": "JPMorgan",
    "NFLX": "Netflix",
    "PFE": "Pfizer",
    "XOM": "Exxon",
    "WMT": "Walmart"
}

In [ ]:
data.loc[data["company"].isna(), "company"] = data.loc[data["company"].isna(), "stocks"].str.split(";").apply(lambda x: ";".join(mapping[i] for i in x if i in mapping))

In [ ]:
data

In [ ]:
stocks = (data["stocks"].fillna("").str.get_dummies(sep=";"))[["AAPL", "TSLA", "CAT", "JPM", "NFLX", "PFE", "XOM", "WMT"]]

In [ ]:
data = pd.concat([stocks, data], axis=1)

In [ ]:
source = data["source"].str.get_dummies()

In [ ]:
data = pd.concat([source, data], axis=1)

In [ ]:
data = data.drop(columns=["stocks", "source"])

In [ ]:
# data.to_csv("data_model.csv", index=False)

In [ ]:
common_cols = [c for c in data.columns if not c.startswith("emb_f_") and not c.startswith("emb_s_")]
f_cols = [c for c in data.columns if c.startswith("emb_f_")]
s_cols = [c for c in data.columns if c.startswith("emb_s_")]

data_f = data[common_cols + f_cols].copy()
data_s = data[common_cols + s_cols].copy()

In [ ]:
data_f = data_f.dropna()
data_s = data_s.dropna()

In [ ]:
# data_f.to_csv("data_f.csv", index=False)
# data_s.to_csv("data_s.csv", index=False)

In [ ]:
data_f.head()

In [ ]:
data_f["webPublicationDate"] = pd.to_datetime(data_f["webPublicationDate"])
data_s["webPublicationDate"] = pd.to_datetime(data_s["webPublicationDate"])

In [ ]:
quotes = pd.read_csv("quotes_all.csv", index_col=0)

In [ ]:
quotes["Date"] = pd.to_datetime(quotes["Date"])

In [ ]:
quotes = quotes.sort_values(["ticker", "Date"]).reset_index(drop=True)

In [ ]:
y_s = data_s[["webPublicationDate"]]
y_f = data_f[["webPublicationDate"]]
X_s = data_s.drop(columns=["webPublicationDate"])
X_f = data_f.drop(columns=["webPublicationDate"])

In [ ]:
y_s["is_summer"] = y_s["webPublicationDate"].apply(lambda x: (x.month > 3 or (x.month == 3 and x.day >= 8)) and (x.month < 11 or (x.month == 11 and x.day < 1)))

In [ ]:
y_s["After Closing"] = y_s["webPublicationDate"].apply(lambda x: x.hour) > (21 - y_s["is_summer"].astype(int))

In [ ]:
y_s.loc[y_s["After Closing"], "Close Price Date"] = y_s["webPublicationDate"].dt.date
y_s.loc[~y_s["After Closing"], "Close Price Date"] = y_s["webPublicationDate"].dt.date - pd.Timedelta(days=1)

In [ ]:
y_s["Close Price Date"] = pd.to_datetime(y_s["Close Price Date"])

In [ ]:
y_s = y_s.merge(quotes, on=["ticker", "Close Price Date"], how="left")